In [2]:
# Imports
import numpy as np
import pandas as pd
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder
from sklearn.compose import ColumnTransformer, make_column_selector as selector
from sklearn.model_selection import GridSearchCV, StratifiedKFold, KFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.multioutput import MultiOutputClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from pandas.api.types import is_datetime64_any_dtype

# 1) Robust datetime expander (no deprecated args; fewer warnings)
def expand_datetime_features(X: pd.DataFrame) -> pd.DataFrame:
    X = X.copy()

    # Drop completely empty columns early (also removes columns like 'Unnamed: 1' if all NaN)
    X = X.dropna(axis=1, how='all')

    # Try to parse object columns to datetimes; keep only if most values parse
    parsed_as_datetime = []
    for col in X.select_dtypes(include='object').columns:
        s = pd.to_datetime(X[col], errors='coerce', utc=True, dayfirst=True)
        if s.notna().mean() >= 0.7:
            # Convert tz-aware to naive UTC
            if getattr(s.dt, 'tz', None) is not None:
                s = s.dt.tz_convert('UTC').dt.tz_localize(None)
            else:
                # Already naive; ensure dtype is datetime64[ns]
                s = s.dt.tz_localize(None)
            X[col] = s
            parsed_as_datetime.append(col)

    # Extract features from all datetime columns
    date_cols = [c for c in X.columns if is_datetime64_any_dtype(X[c])]
    for c in date_cols:
        s = X[c]
        X[f'{c}_year'] = s.dt.year.astype('float64')
        X[f'{c}_month'] = s.dt.month.astype('float64')
        X[f'{c}_day'] = s.dt.day.astype('float64')
        X[f'{c}_dow'] = s.dt.dayofweek.astype('float64')
        X[f'{c}_is_month_end'] = s.dt.is_month_end.astype('float64')

    if date_cols:
        X = X.drop(columns=date_cols)

    return X

# 2) Version-safe OneHotEncoder instance
try:
    ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)  # sklearn >= 1.2
except TypeError:
    ohe = OneHotEncoder(handle_unknown='ignore', sparse=False)         # older sklearn

# 3) Preprocess transformer
preprocess = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), selector(dtype_include=np.number)),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('ohe', ohe)
    ]), selector(dtype_exclude=np.number))
])

dates_tf = FunctionTransformer(expand_datetime_features)

# 4) Load data
df = pd.read_excel(r'C:\AI_ML\Cursor_Projects\Project_RAG\HyperParameter_Billable_Customer_Projs-2025-10-23.xlsx')

# Drop completely empty columns early to avoid imputer warnings on all-NaN features (e.g., 'Unnamed: 1')
df = df.dropna(axis=1, how='all')

target_cols = ['Revenue Status', 'Schedule Status', 'Overall Status', 'Client Blockage Status']

# Ensure no missing targets (a common cause of NaN CV scores)
df = df.dropna(subset=target_cols)

X = df.drop(columns=target_cols)
y = df[target_cols]

# 5) Pipeline
pipe = Pipeline([
    ('dates', dates_tf),
    ('prep', preprocess),
    ('clf', MultiOutputClassifier(HistGradientBoostingClassifier(random_state=42)))
])

# 6) Param grid
param_grid = {
    'clf__estimator__max_iter': [300, 500],
    'clf__estimator__learning_rate': [0.03, 0.05, 0.1],
    'clf__estimator__max_depth': [5, 7],
    'clf__estimator__min_samples_leaf': [10, 20],
    'clf__estimator__l2_regularization': [0.0, 0.1, 0.3],
}

# 7) Build CV splits with robust fallback
def make_cv_splits(X, y, n_splits=5, random_state=42):
    # Combine multi-output labels into a single stratification label
    strat_labels = y.astype(str).agg('|'.join, axis=1)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    try:
        return list(skf.split(X, strat_labels))
    except ValueError:
        # Fallback when any class has < n_splits samples
        kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
        return list(kf.split(X))

cv_splits = make_cv_splits(X, y, n_splits=5, random_state=42)

# 8) Grid search
grid = GridSearchCV(
    pipe,
    param_grid,
    cv=cv_splits,
    n_jobs=-1,
    verbose=2,
    scoring='accuracy',
    # Uncomment to surface the exact error if any fit still fails:
    # error_score='raise'
)

grid.fit(X, y)
print("Best params:", grid.best_params_)

C:\Users\kumarni\AppData\Roaming\Python\Python313\site-packages\sklearn\model_selection\_split.py:811: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


Fitting 5 folds for each of 72 candidates, totalling 360 fits


C:\Users\kumarni\AppData\Roaming\Python\Python313\site-packages\sklearn\model_selection\_search.py:1135: UserWarning: One or more of the test scores are non-finite: [nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan]
  warnings.warn(
C:\Users\kumarni\AppData\Local\Temp\1\ipykernel_26004\2914976888.py:23: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  s = pd.to_datetime(X[col], errors='coerce', utc=True, dayfirst=True)
C:\Users\kumarni\AppData\Local\Temp\1\ipykernel_26004\2914976888.py:23: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsin

Best params: {'clf__estimator__l2_regularization': 0.0, 'clf__estimator__learning_rate': 0.03, 'clf__estimator__max_depth': 5, 'clf__estimator__max_iter': 300, 'clf__estimator__min_samples_leaf': 10}
